<a href="https://colab.research.google.com/github/Pandabear-67/Collision-tracker/blob/main/LSTM_Autoencoder/LSTM_autoencoder_interpretation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
base_path = "/content/drive/MyDrive/accident_files/notebooks"
import sys
sys.path.append(base_path)

In [ ]:
import cv2

class VideoFrameExtractor:
    def __init__(self, video_path):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)

        if not self.cap.isOpened():
            raise ValueError(f"Could not open video: {video_path}")

        # FPS (fallback-safe)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        if self.fps is None or self.fps <= 0:
            self.fps = 30.0

        # total frames (optional, may be 0 for some codecs)
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def get_frames(self):
        """
        Generator yielding:
            frame_idx (int), frame (BGR image as numpy array)
        """
        frame_idx = 0

        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            yield frame_idx, frame
            frame_idx += 1

        self.cap.release()

    def reset(self):
        """
        Rewinds video for re-use.
        """
        self.cap.release()
        self.cap = cv2.VideoCapture(self.video_path)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from tracking_flow_pipeline_experiment import DetectorTracker, OpticalFlow, FeatureBuilder, SequenceBuilder, HDF5Writer, pool_flow

def detect_accident(
    video_path,
    lstm_autoencoder,
    detector_tracker,
    optical_flow,
    device,
    fps=30,
    threshold=0.01,
    alpha=0.6
):
    extractor = VideoFrameExtractor(video_path)
    fps = extractor.fps if extractor.fps > 0 else fps

    feat_builder = FeatureBuilder()
    seq_builder = SequenceBuilder(seq_len=16, max_objects=50, feat_dim=13)


    prev_frame = None

    frame_indices = []
    times = []
    anomaly_scores = []

    for frame_idx, frame in extractor.get_frames():

        # -----------------------------
        # 1. Detection + tracking
        # -----------------------------
        boxes, ids = detector_tracker.update(frame)

        if prev_frame is not None:

            # -----------------------------
            # 2. Optical flow
            # -----------------------------
            flow = optical_flow.compute_flow(prev_frame, frame)

            # -----------------------------
            # 3. Feature extraction
            # -----------------------------
            obj_feats = feat_builder.bbox_to_features(boxes, ids, frame.shape)
            flow_feats = pool_flow(flow, boxes)

            if len(obj_feats) == 0:
                full_feats = np.zeros((1, 13), dtype=np.float32)
            else:
                n = min(len(obj_feats), len(flow_feats))
                full_feats = np.concatenate(
                    [obj_feats[:n], flow_feats[:n]],
                    axis=1
                )

            # -----------------------------
            # 4. Build sequence
            # -----------------------------
            seq = seq_builder.add_frame(full_feats)

            # -----------------------------
            # 5. LSTM autoencoder inference
            # -----------------------------
            if seq is not None:
                seq_flat = seq.reshape(1, 16, 50*13)

                recon = lstm_autoencoder.predict(seq_flat, verbose=0)

                error = np.mean((seq_flat - recon)** 2)

                frame_indices.append(frame_idx)
                times.append(frame_idx / fps)
                anomaly_scores.append(error)

        prev_frame = frame

    # -----------------------------
    # 6. Detect first anomaly spike
    # -----------------------------
    anomaly_scores = np.array(anomaly_scores)

    # smoothing (important for stability)
    smooth_scores = np.copy(anomaly_scores)
    for i in range(1, len(smooth_scores)):
        smooth_scores[i] = alpha * smooth_scores[i] + (1 - alpha) * smooth_scores[i - 1]

    spike_idx = None
    # for i, s in enumerate(smooth_scores):
    #     if s > threshold:
    #         spike_idx = i
    #         break

    anomaly_scores = np.array(anomaly_scores)

    # smoothing
    smooth_scores = np.copy(anomaly_scores)
    for i in range(1, len(smooth_scores)):
        smooth_scores[i] = alpha * smooth_scores[i] + (1 - alpha) * smooth_scores[i - 1]

    # find peak
    peak_idx = np.argmax(smooth_scores)
    peak_score = smooth_scores[peak_idx]

    # optional: require it to exceed threshold
    if peak_score > threshold:
        accident_frame = frame_indices[peak_idx]
        accident_time = times[peak_idx]
        score = peak_score

        print("ACCIDENT DETECTED (PEAK)")
        print("Frame:", accident_frame)
        print("Time (s):", accident_time)
        print("Score:", score)
    else:
        accident_frame, accident_time, score = -1, None, None [l]
        print("No anomaly above threshold")

    if spike_idx is not None:
        accident_frame = frame_indices[spike_idx]
        accident_time = times[spike_idx]
        score = smooth_scores[spike_idx]

        print("ACCIDENT DETECTED")
        print("Frame:", accident_frame)
        print("Time (s):", accident_time)
        print("Score:", score)
    else:
        accident_frame, accident_time, score = -1, None, None
        print("No accident detected")

    # -----------------------------
    # 7. Visualization
    # -----------------------------
    plt.plot(times, smooth_scores)
    plt.axhline(threshold, color="g")
    if spike_idx is not None:
        plt.axvline(accident_time, color="r", label="Peak anomaly")
    plt.xlabel("Time (s)")
    plt.ylabel("Reconstruction error")
    plt.title("Accident Detection Signal")
    plt.show()

    return accident_frame, accident_time, score

In [ ]:
lstm_autoencoder = tf.keras.models.load_model("/content/drive/MyDrive/accident_files/models/accident_autoencoder.keras")
detector = DetectorTracker("/content/drive/MyDrive/accident_files/models/yolo26n_apr5.pt")
flow = OpticalFlow("/content/drive/MyDrive/accident_files/RAFT/models/raft-sintel.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
detect_accident(video_path="/content/drive/MyDrive/accident_files/videos/1D0KxP4drTU_00.mp4",lstm_autoencoder=lstm_autoencoder, detector_tracker=detector, optical_flow=flow, device=device)